In [ ]:
print("Nama  : Nur Bian Aziza")
print("NPM   : 11124490")
print("Kelas : 3KA31")

Nama  : Nur Bian Aziza
NPM   : 11124490
Kelas : 3KA31


**1. Import library & membaca dataset**

In [ ]:
import pandas as pd

# Membaca dataset
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

# Menampilkan 5 data pertama
display(df.head())

# Menampilkan jumlah baris dan kolom
print("Ukuran dataset:", df.shape)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


Ukuran dataset: (5030, 9)


**2. Memeriksa missing value**

In [ ]:
print("Jumlah missing value setiap kolom:")
print(df.isnull().sum())

Jumlah missing value setiap kolom:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64


3. **Menangani missing value**

Missing value pada **kode_kasir** diisi dengan TIDAK DIKETAHUI. Baris tidak dihapus karena informasi kode kasir tidak diperlukan untuk menghitung pendapatan dan transaksi tetap perlu dipertahankan.

In [ ]:
df['kode_kasir'] = df['kode_kasir'].fillna('TIDAK DIKETAHUI')

print("Missing value setelah penanganan:")
print(df.isnull().sum())

Missing value setelah penanganan:
id_transaksi     0
tanggal          0
kode_gerai       0
kode_kasir       0
kode_produk      0
jumlah           0
harga_satuan     0
diskon_persen    0
status           0
dtype: int64


**4. Memeriksa dan menghapus data duplikat**

In [ ]:
# Mengecek jumlah data duplikat
print("Data duplikat:", df.duplicated().sum())

# Menghapus data duplikat
df = df.drop_duplicates()

# Mengecek jumlah data setelah penghapusan
print("Jumlah data setelah dibersihkan:", len(df))

Data duplikat: 30
Jumlah data setelah dibersihkan: 5000


**5. Membersihkan dan Mengubah Tipe Data**

Ada beberapa masalah format di data ini:
- tanggal: format tanggal tidak seragam, seperti `2024-12-28`, `26/03/2025`, dan `23-12-2024`
- harga_satuan: format angka berbeda, seperti 7500,00 dan 13.500.
- jumlah dan diskon_persen: perlu diubah menjadi tipe numerik.
Proses ini dilakukan agar data lebih seragam dan bisa digunakan untuk perhitungan pendapatan.

In [ ]:
# Mengubah format tanggal
df['tanggal'] = pd.to_datetime(
    df['tanggal'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

# Membersihkan format harga satuan
df.loc[:, 'harga_satuan'] = pd.to_numeric(
    df['harga_satuan'].astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

# Mengubah jumlah dan diskon menjadi numerik
df.loc[:, 'jumlah'] = pd.to_numeric(df['jumlah'], errors='coerce')
df.loc[:, 'diskon_persen'] = pd.to_numeric(df['diskon_persen'], errors='coerce')

# Menampilkan tipe data
print(df.dtypes)

id_transaksi             object
tanggal          datetime64[ns]
kode_gerai               object
kode_kasir               object
kode_produk              object
jumlah                    int64
harga_satuan             object
diskon_persen             int64
status                   object
pendapatan               object
dtype: object


**6. Hitung pendapatan per tahun**

Rumus: **pendapatan = jumlah x harga_satuan x (1 - diskon_persen / 100)**

In [ ]:
# Menghitung pendapatan setelah diskon
df.loc[:, 'pendapatan'] = (
    df['jumlah'] *
    df['harga_satuan'] *
    (1 - df['diskon_persen'] / 100)
)

# Mengambil tahun dari tanggal transaksi
df.loc[:, 'tahun'] = df['tanggal'].dt.year

# Menampilkan hasil perhitungan
display(df[['tanggal', 'jumlah', 'harga_satuan',
            'diskon_persen', 'status', 'pendapatan']].head())

,tanggal,jumlah,harga_satuan,diskon_persen,status,pendapatan
0,2025-06-02,4,130000,0,SELESAI,520000.0
1,2024-12-28,1,210000,0,SELESAI,210000.0
2,2024-09-04,3,75000,0,SELESAI,225000.0
3,2025-07-30,2,65000,0,SELESAI,130000.0
4,2025-03-26,1,60000,0,SELESAI,60000.0


**7. Menghitung total pendapatan tahun 2024 dan 2025**


In [ ]:
# Memilih transaksi yang berstatus SELESAI
transaksi_selesai = df[
    (df['status'].str.strip().str.upper() == 'SELESAI') &
    (df['tahun'].isin([2024, 2025]))
]

# Menghitung total pendapatan berdasarkan tahun
total_pendapatan = transaksi_selesai.groupby('tahun')['pendapatan'].sum()

# Mengambil total pendapatan masing-masing tahun
pendapatan_2024 = total_pendapatan.get(2024, 0)
pendapatan_2025 = total_pendapatan.get(2025, 0)

# Menampilkan hasil akhir
print("Total Pendapatan 2024: Rp" + format(pendapatan_2024, ',.0f').replace(',', '.'))
print("Total Pendapatan 2025: Rp" + format(pendapatan_2025, ',.0f').replace(',', '.'))

Total Pendapatan 2024: Rp3.862.373.750
Total Pendapatan 2025: Rp2.570.851.000
